# 09 - Instruction Tuning - llama3.2

### setup

In [1]:
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton unsloth_zoo
!pip install --no-deps cut_cross_entropy unsloth
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 20.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.9.2 requires structlog>=24.1.0, which is not installed.
unsloth 2026.9.2 requires tyro, which is not installed.
unsloth-zoo 2026.9.1 requires msgspec, which is not installed.
unsloth-zoo 2026.9.1 requires tyro; sys_platform != "darwin" or platform_machine != "ar

In [2]:
import json
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Konfigurasi

In [3]:
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_Faiss")
 
HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
OUTPUT_DIR = "/kaggle/working"
 
MODEL_KEY = "gemma2"
MODEL_NAME = "unsloth/gemma-2-2b-it-bnb-4bit"
SYSTEM_ROLE = False

In [4]:
DRY_RUN = False  # True: subset kecil data + steps terbatas, untuk tes cepat di Kaggle. False: training penuh (RunPod)
DRY_RUN_SAMPLES = 40
DRY_RUN_MAX_STEPS = 10
 
TIMING_TEST = False  # True: jalan 50 steps di data PENUH untuk mengukur waktu/step nyata, sebelum commit full training
TIMING_TEST_MAX_STEPS = 50
 
MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
 
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 3
WARMUP_STEPS = 0
MAX_GRAD_NORM = 0.3
SEED = 42
 
INSTRUCTION_PART = "<start_of_turn>user\n"
RESPONSE_PART = "<start_of_turn>model\n"
 
SYSTEM_PROMPT = (
    "Anda adalah asisten yang menjawab pertanyaan seputar publikasi statistik BPS "
    "berdasarkan konteks yang diberikan. Kutip bagian konteks yang relevan sebelum menjawab."
)

## 2. Load dataset

In [5]:
def load_formatted_split(split_name):
    config_name = f"{MODEL_KEY}_formatted_{split_name}"
    return load_dataset(HF_REPO, config_name, split="train", token=HF_TOKEN)
 
 
train_ds = load_formatted_split("train")
val_ds = load_formatted_split("val")
 
if DRY_RUN:
    train_ds = train_ds.select(range(min(DRY_RUN_SAMPLES, len(train_ds))))
    val_ds = val_ds.select(range(min(DRY_RUN_SAMPLES // 4, len(val_ds))))
 
print(f"[{'DRY_RUN' if DRY_RUN else 'FULL'}] train: {len(train_ds)} | val: {len(val_ds)}")

README.md: 0.00B [00:00, ?B/s]

gemma2_formatted_train/train-00000-of-00(…):   0%|          | 0.00/1.92M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6516 [00:00<?, ? examples/s]

gemma2_formatted_val/train-00000-of-0000(…):   0%|          | 0.00/315k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1043 [00:00<?, ? examples/s]

[FULL] train: 6516 | val: 1043


## 3. Load model + LoRA

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

==((====))==  Unsloth 2026.9.2: Fast Gemma2 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 2.042 GiB
no_split classes   : ['Gemma2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.494 GiB
activation reserve : 11.706 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  0.943 GiB  free 12.009 GiB  reserve 11.706 GiB
  cuda:1  budget  13.00 GiB  weights  1.099 GiB  free 11.898 GiB  reserve 11.403 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.440 -> 12.996 GiB (memory t

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.2 patched 26 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## 4. Main Program (Training)

### Training ARGS

In [8]:
autocast_dtype_flags = {
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
}
 
training_args = SFTConfig(
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=(DRY_RUN_MAX_STEPS if DRY_RUN else (TIMING_TEST_MAX_STEPS if TIMING_TEST else -1)),
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="constant",
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=1 if DRY_RUN else 10,
    seed=SEED,
    output_dir=f"{OUTPUT_DIR}/{MODEL_KEY}_checkpoints",
    eval_strategy="steps",
    eval_steps=5 if DRY_RUN else 50,
    save_strategy="steps",
    save_steps=5 if DRY_RUN else 50,
    save_total_limit=3,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    **autocast_dtype_flags,
)

### Trainer

In [9]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)
 
trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_PART,
    response_part=RESPONSE_PART,
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6516 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1043 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/6516 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1043 [00:00<?, ? examples/s]

## 5. Save dan Push hasil ke HF

In [10]:
RESUME_FROM_CHECKPOINT = False  # True: lanjutkan dari checkpoint terakhir di output_dir kalau sesi sebelumnya terputus

checkpoint_dir = f"{OUTPUT_DIR}/{MODEL_KEY}_checkpoints"
resume_path = None
if RESUME_FROM_CHECKPOINT and os.path.isdir(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        resume_path = os.path.join(checkpoint_dir, sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1])
        print(f"Melanjutkan dari: {resume_path}")

trainer_stats = trainer.train()
print(trainer_stats)
 
if TIMING_TEST:
    steps_done = trainer.state.global_step
    total_runtime = trainer_stats.metrics.get("train_runtime")
    sec_per_step = total_runtime / steps_done
    full_train_len = len(load_formatted_split("train"))
    steps_per_epoch = full_train_len / (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
    total_steps_full = int(steps_per_epoch * NUM_TRAIN_EPOCHS)
    estimated_full_runtime_min = (sec_per_step * total_steps_full) / 60
    print(f"[TIMING_TEST] {sec_per_step:.2f} detik/step, estimasi total training penuh: {estimated_full_runtime_min:.1f} menit ({total_steps_full} steps)")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 6,516 | Num Epochs = 3 | Total steps = 1,224
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.605959,0.586530
100,0.516120,0.532535
150,0.512292,0.510041
200,0.476727,0.494694
250,0.488645,0.483928
300,0.469185,0.476580
350,0.478380,0.468103
400,0.429391,0.461658
450,0.395961,0.460074
500,0.434724,0.455453


Filter:   0%|          | 0/1043 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma2_checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/gemma2_checkpoints/checkpoint-50.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma2_checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/gemma2_checkpoints/checkpoint-100.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma2_checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/gemma2_checkpoints/checkpoint-150.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma2_checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/gemma2_checkpoints/checkpoint-200.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/wor

TrainOutput(global_step=1224, training_loss=0.43756067402222576, metrics={'train_runtime': 14399.5336, 'train_samples_per_second': 1.358, 'train_steps_per_second': 0.085, 'total_flos': 9.066592861861478e+16, 'train_loss': 0.43756067402222576, 'epoch': 3.0})


In [11]:
adapter_dir = f"{OUTPUT_DIR}/{MODEL_KEY}_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
 
if not DRY_RUN and not TIMING_TEST:
    model.push_to_hub(f"Makaareeem/{MODEL_KEY}-qlora-adapter", token=HF_TOKEN)
    tokenizer.push_to_hub(f"Makaareeem/{MODEL_KEY}-qlora-adapter", token=HF_TOKEN)
else:
    print("[DRY_RUN/TIMING_TEST] skip push_to_hub — hanya disimpan lokal untuk cek logic")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/gemma2_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/gemma2_adapter.


README.md:   0%|          | 0.00/554 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Makaareeem/gemma2-qlora-adapter


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp19w3czt_/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /tmp/tmp19w3czt_.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## 6. Ringkasan

In [12]:
summary = {
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
    "train_loss_history": [log for log in trainer.state.log_history if "loss" in log],
    "eval_loss_history": [log for log in trainer.state.log_history if "eval_loss" in log],
    "total_train_time_sec": trainer_stats.metrics.get("train_runtime"),
    "hyperparameters": {
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_TRAIN_EPOCHS,
        "per_device_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    },
}
 
with open(f"{OUTPUT_DIR}/{MODEL_KEY}_training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
 
print(json.dumps(summary["hyperparameters"], indent=2))
print(f"Final train loss: {summary['train_loss_history'][-1] if summary['train_loss_history'] else None}")
print(f"Final eval loss: {summary['eval_loss_history'][-1] if summary['eval_loss_history'] else None}")

{
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.1,
  "learning_rate": 2e-05,
  "epochs": 3,
  "per_device_batch_size": 4,
  "gradient_accumulation_steps": 4,
  "effective_batch_size": 16
}
Final train loss: {'loss': 0.37800133228302, 'grad_norm': 2.2446694374084473, 'learning_rate': 2e-05, 'epoch': 2.9920196439533457, 'step': 1220}
Final eval loss: {'eval_loss': 0.44047707319259644, 'eval_runtime': 196.6721, 'eval_samples_per_second': 5.303, 'eval_steps_per_second': 1.327, 'epoch': 2.942909760589319, 'step': 1200}
